In [1]:
import os
import pandas as pd
import json
from pathlib import Path

## **Section-01**: Data Integrity Check

In [2]:
DATASET_ROOT = "/kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset"

TRAIN_META_PATH = f"{DATASET_ROOT}/extracted_text/train_metadata.csv"
TEST_META_PATH = f"{DATASET_ROOT}/extracted_text/test_metadata.csv"

TRAIN_FRAME_PATH = f"{DATASET_ROOT}/extracted_frames/train"
TEST_FRAME_PATH = f"{DATASET_ROOT}/extracted_frames/test"

TRAIN_ANNOT_PATH = f"{DATASET_ROOT}/annotations/train_annotations.json"
TEST_ANNOT_PATH = f"{DATASET_ROOT}/annotations/test_annotations.json"

In [3]:
train_meta = pd.read_csv(TRAIN_META_PATH)
test_meta = pd.read_csv(TEST_META_PATH)

print("Train samples:", len(train_meta))
print("Test samples:", len(test_meta))

train_meta.head()

Train samples: 1600
Test samples: 800


,video_id,has_audio,transcript,ocr_text,category,subcategory
0,IF_001,True,আপনি সিদ্ধান্ত নিয়েছেন একটি নতুন অ্যাপ্লিকেশন...,OR EVEN TAKA ON YOUR FIRST CRAZY TIME,misleading,identity_fabrication
1,IF_002,True,"With a small investment of 30,500 Bangladeshi ...",YES WITH A SMALL INVESTMENT OF BANGLADESH TAKA...,misleading,identity_fabrication
2,IF_003,True,"If they close this page, they can't come back ...",IF THEY CLOSE THIS PAGE AS WILL THEIR CHANCE T...,misleading,identity_fabrication
3,IF_004,True,We tested our product with a small group of vo...,AND EACH OF THEM EARNED OVER BANGLADESHI TAKA ...,misleading,identity_fabrication
4,IF_005,True,I am ready to give you your money back from my...,MONEY BACK FROM MY POCKET BANGLADESHI TAKA,misleading,identity_fabrication


In [4]:
print("\nTrain Category Distribution")
print(train_meta["category"].value_counts())

print("\nTrain Subcategory Distribution")
print(train_meta["subcategory"].value_counts())


Train Category Distribution
category
misleading    800
safe          800
Name: count, dtype: int64

Train Subcategory Distribution
subcategory
identity_fabrication                200
perception_manipulation             200
scientifically_unrealistic_scene    200
surreal_content                     200
Name: count, dtype: int64


In [5]:
frame_root = Path(TRAIN_FRAME_PATH)

sample_video = train_meta.iloc[0]["video_id"]

sample_frame_dir = frame_root / sample_video

frames = list(sample_frame_dir.glob("*.jpg"))

print("Sample video:", sample_video)
print("Frame count:", len(frames))
print("Example frame:", frames[0])

Sample video: IF_001
Frame count: 16
Example frame: /kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset/extracted_frames/train/IF_001/frame_10.jpg


In [6]:
import random

sample_ids = random.sample(list(train_meta["video_id"]), 5)

for vid in sample_ids:
    
    frame_dir = Path(TRAIN_FRAME_PATH) / vid
    frame_count = len(list(frame_dir.glob("*.jpg")))
    
    print(f"{vid} -> {frame_count} frames")

SAFE_475 -> 16 frames
IF_058 -> 16 frames
IF_034 -> 16 frames
IF_084 -> 16 frames
SAFE_218 -> 16 frames


In [7]:
train_meta[["video_id", "has_audio", "transcript"]].head(10)

,video_id,has_audio,transcript
0,IF_001,True,আপনি সিদ্ধান্ত নিয়েছেন একটি নতুন অ্যাপ্লিকেশন...
1,IF_002,True,"With a small investment of 30,500 Bangladeshi ..."
2,IF_003,True,"If they close this page, they can't come back ..."
3,IF_004,True,We tested our product with a small group of vo...
4,IF_005,True,I am ready to give you your money back from my...
5,IF_006,True,"So far, there has not been a single person who..."
6,IF_007,True,"I've decided to create a new application, Craz..."
7,IF_008,False,NaN
8,IF_009,True,We the people stand united now
9,IF_010,False,NaN


In [8]:
train_meta[["video_id", "ocr_text"]].head(10)

,video_id,ocr_text
0,IF_001,OR EVEN TAKA ON YOUR FIRST CRAZY TIME
1,IF_002,YES WITH A SMALL INVESTMENT OF BANGLADESH TAKA...
2,IF_003,IF THEY CLOSE THIS PAGE AS WILL THEIR CHANCE T...
3,IF_004,AND EACH OF THEM EARNED OVER BANGLADESHI TAKA ...
4,IF_005,MONEY BACK FROM MY POCKET BANGLADESHI TAKA
5,IF_006,NO SPECIAL SKILLS ARE REQUIRED IF THEY WATCH T...
6,IF_007,MANY PLAYERS HAVE BOUGHT CARS AND HOUSES FOR T...
7,IF_008,NaN
8,IF_009,WE THE PEOPLE STAND NO NO KINGS NO KING S NO
9,IF_010,NaN


In [9]:
with open(TRAIN_ANNOT_PATH) as f:
    train_annotations = json.load(f)

with open(TEST_ANNOT_PATH) as f:
    test_annotations = json.load(f)

print("Train annotations:", len(train_annotations))
print("Test annotations:", len(test_annotations))

Train annotations: 1600
Test annotations: 800


In [10]:
print(train_annotations[0])


print(train_annotations[956])

{'video_id': 'IF_001', 'category': 'misleading', 'subcategory': 'identity_fabrication', 'audio_available': True, 'visual_description': 'An elderly man with gray hair is seated on an ornate white carved chair, speaking directly toward the camera. The background features wooden interior decor. The scene remains static, and subtitle-style text appears at the bottom of the screen.', 'audio_transcript': "I've decided to create a new application. Crazy time. By investing just 5 taka, you can win 50,000 or even 100,000 taka on your first day.", 'ocr_text': ['due to the current crisis in the country', "I've decided to create a new application", 'crazy time', 'by investing just 5 taka you can win 50,000'], 'scene_summary': 'A man appears to be giving an announcement while subtitles promote a financial scheme claiming that a 5-taka investment can generate 50,000–100,000 taka. These claims are unrealistic and represent deceptive financial misinformation.', 'explanation': {'visual_cues': 'Static i

In [11]:
print("\nDATASET SUMMARY")
print("--------------")

print("Train Videos:", len(train_meta))
print("Test Videos:", len(test_meta))

print("Train Annotations:", len(train_annotations))
print("Test Annotations:", len(test_annotations))

print("\nDataset integrity check completed successfully.")


DATASET SUMMARY
--------------
Train Videos: 1600
Test Videos: 800
Train Annotations: 1600
Test Annotations: 800

Dataset integrity check completed successfully.


## **Section-02**: Multimodal DataLoader

In [12]:
!pip install transformers timm --quiet

In [13]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

from PIL import Image
import numpy as np
import pandas as pd
from pathlib import Path

In [14]:
# Label Mapping

label_map = {
    "safe":0,
    "identity_fabrication":1,
    "perception_manipulation":2,
    "scientifically_unrealistic_scene":3,
    "surreal_content":4
}

In [15]:
# Image Transform

image_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

In [16]:
class CrossModalDataset(Dataset):

    def __init__(self, metadata, frame_root):

        self.metadata = metadata
        self.frame_root = Path(frame_root)

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):

        row = self.metadata.iloc[idx]

        video_id = row["video_id"]

        frame_dir = self.frame_root / video_id

        frames = sorted(list(frame_dir.glob("*.jpg")))[:16]

        images = []

        for f in frames:
            img = Image.open(f).convert("RGB")
            img = image_transform(img)
            images.append(img)

        images = torch.stack(images)

        transcript = str(row["transcript"])
        ocr_text = str(row["ocr_text"])

        text_input = transcript + " " + ocr_text

        if row["category"] == "safe":
            label = label_map["safe"]
        else:
            label = label_map[row["subcategory"]]

        return {
            "images": images,
            "text": text_input,
            "label": label,
            "video_id": video_id
        }

In [17]:
train_dataset = CrossModalDataset(
    train_meta,
    TRAIN_FRAME_PATH
)

test_dataset = CrossModalDataset(
    test_meta,
    TEST_FRAME_PATH
)

print("Train dataset:", len(train_dataset))
print("Test dataset:", len(test_dataset))

Train dataset: 1600
Test dataset: 800


In [18]:
sample = train_dataset[0]

print("Video ID:", sample["video_id"])
print("Image tensor shape:", sample["images"].shape)
print("Text:", sample["text"][:120])
print("Label:", sample["label"])

Video ID: IF_001
Image tensor shape: torch.Size([16, 3, 224, 224])
Text: আপনি সিদ্ধান্ত নিয়েছেন একটি নতুন অ্যাপ্লিকেশন ক্রেজি টাইম তৈরি করার। মাত্র ৫ টাকা বিনিয়োগ করে আপনি প্রথম দিনেই ৫০,০০০ 
Label: 1


In [19]:
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=2, shuffle=False, num_workers=2, pin_memory=True)
print("Dataloader ready.")

Dataloader ready.


## Section-03: Multimodal Neural Network

In [20]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from transformers import CLIPModel, AutoTokenizer, AutoModel
from torch.cuda.amp import autocast, GradScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
roberta = AutoModel.from_pretrained("roberta-base").to(device)

# Freeze encoders to save memory
for p in clip.parameters():
    p.requires_grad = False
for p in roberta.parameters():
    p.requires_grad = False

clip.eval()
roberta.eval()

print("Encoders loaded + frozen.")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Encoders loaded + frozen.


## Section-04: Training Setup

In [21]:
def clip_vision_embed(pixel_values):
    """
    Always returns a TENSOR embedding, compatible across HF versions.
    """
    outputs = clip.vision_model(pixel_values=pixel_values)

    if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
        return outputs.pooler_output  # (B, hidden)

    # fallback
    return outputs.last_hidden_state.mean(dim=1)  # (B, hidden)

In [22]:
# Take one batch to detect fusion dimension safely
batch0 = next(iter(train_loader))
images0 = batch0["images"]                      # (B,16,3,224,224)
texts0  = list(batch0["text"])

B = images0.size(0)
images0 = images0.view(-1,3,224,224).to(device) # (B*16,3,224,224)

with torch.no_grad():
    img_feat0 = clip_vision_embed(images0)      # (B*16, Dimg)
img_feat0 = img_feat0.view(B,16,-1).mean(dim=1) # (B, Dimg)

with torch.no_grad():
    toks0 = tokenizer(texts0, padding=True, truncation=True, return_tensors="pt").to(device)
    txt_out0 = roberta(**toks0)
    txt_feat0 = txt_out0.last_hidden_state[:,0,:]  # (B,768)

fusion_dim = img_feat0.shape[1] + txt_feat0.shape[1]
print("Detected dims -> image:", img_feat0.shape[1], "text:", txt_feat0.shape[1], "fusion:", fusion_dim)

Detected dims -> image: 768 text: 768 fusion: 1536


In [23]:
class FusionClassifier(nn.Module):
    def __init__(self, fusion_dim, num_classes=5):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, images, texts):
        b = images.size(0)
        images = images.view(-1, 3, 224, 224).to(device)

        with torch.no_grad():
            img_feat = clip_vision_embed(images)          # (B*16, Dimg)
        img_feat = img_feat.view(b, 16, -1).mean(dim=1)   # (B, Dimg)

        with torch.no_grad():
            toks = tokenizer(list(texts), padding=True, truncation=True, return_tensors="pt").to(device)
            txt_out = roberta(**toks)
            txt_feat = txt_out.last_hidden_state[:, 0, :] # (B,768)

        fusion = torch.cat([img_feat, txt_feat], dim=1)   # (B, fusion_dim)
        return self.classifier(fusion)

model = FusionClassifier(fusion_dim=fusion_dim, num_classes=5).to(device)
print("Fusion model created.")

Fusion model created.


In [24]:
criterion = nn.CrossEntropyLoss()

# only classifier has trainable parameters
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)

scaler = GradScaler()
print("Optimizer created. Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

Optimizer created. Trainable params: 789509


/tmp/ipykernel_24/311204766.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [25]:
EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for batch in tqdm(train_loader):
        images = batch["images"]
        texts = batch["text"]
        labels = batch["label"].to(device)

        optimizer.zero_grad(set_to_none=True)

        with autocast():
            outputs = model(images, texts)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} Loss: {total_loss/len(train_loader):.4f}")

  0%|          | 0/800 [00:00<?, ?it/s]/tmp/ipykernel_24/3446227605.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
100%|██████████| 800/800 [01:16<00:00, 10.50it/s]


Epoch 1/5 Loss: 0.5611


100%|██████████| 800/800 [00:56<00:00, 14.17it/s]


Epoch 2/5 Loss: 0.2971


100%|██████████| 800/800 [00:56<00:00, 14.11it/s]


Epoch 3/5 Loss: 0.2143


100%|██████████| 800/800 [00:56<00:00, 14.12it/s]


Epoch 4/5 Loss: 0.1557


100%|██████████| 800/800 [00:56<00:00, 14.15it/s]

Epoch 5/5 Loss: 0.1620


In [26]:
from torch.amp import autocast

In [27]:
!pip install scikit-learn --quiet

In [28]:
import numpy as np
import torch
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd

In [29]:
id2label = {
    0: "safe",
    1: "identity_fabrication",
    2: "perception_manipulation",
    3: "scientifically_unrealistic_scene",
    4: "surreal_content"
}
label2id = {v:k for k,v in id2label.items()}

In [30]:
from torch.amp import autocast

model.eval()

all_preds = []
all_labels = []
all_video_ids = []

with torch.no_grad():
    for batch in tqdm(test_loader):

        images = batch["images"]
        texts  = batch["text"]
        labels = batch["label"].to(device)

        with autocast('cuda'):
            logits = model(images, texts)

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_video_ids.extend(batch["video_id"])

acc = accuracy_score(all_labels, all_preds)
print("TEST Accuracy:", acc)

100%|██████████| 400/400 [00:37<00:00, 10.65it/s]

TEST Accuracy: 0.745


In [31]:
print("\nClassification Report (TEST):\n")
print(classification_report(
    all_labels, all_preds,
    target_names=[id2label[i] for i in range(5)],
    digits=4
))


Classification Report (TEST):

                                  precision    recall  f1-score   support

                            safe     0.8736    0.7600    0.8128       400
            identity_fabrication     0.5270    0.7800    0.6290       100
         perception_manipulation     0.5984    0.7600    0.6696       100
scientifically_unrealistic_scene     0.6765    0.6900    0.6832       100
                 surreal_content     0.9200    0.6900    0.7886       100

                        accuracy                         0.7450       800
                       macro avg     0.7191    0.7360    0.7166       800
                    weighted avg     0.7770    0.7450    0.7527       800



In [32]:
cm = confusion_matrix(all_labels, all_preds)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{id2label[i]}" for i in range(5)],
    columns=[f"pred_{id2label[i]}" for i in range(5)]
)

cm_df

,pred_safe,pred_identity_fabrication,pred_perception_manipulation,pred_scientifically_unrealistic_scene,pred_surreal_content
true_safe,304,47,19,25,5
true_identity_fabrication,14,78,8,0,0
true_perception_manipulation,5,17,76,2,0
true_scientifically_unrealistic_scene,3,5,22,69,1
true_surreal_content,22,1,2,6,69


In [33]:
pred_df = pd.DataFrame({
    "video_id": all_video_ids,
    "true_label": [id2label[x] for x in all_labels],
    "pred_label": [id2label[x] for x in all_preds],
})

pred_df.to_csv("test_predictions_v1.csv", index=False)
print("Saved: test_predictions_v1.csv")
pred_df.head()

Saved: test_predictions_v1.csv


,video_id,true_label,pred_label
0,TEST_IF_001,identity_fabrication,identity_fabrication
1,TEST_IF_002,identity_fabrication,identity_fabrication
2,TEST_IF_003,identity_fabrication,identity_fabrication
3,TEST_IF_004,identity_fabrication,identity_fabrication
4,TEST_IF_005,identity_fabrication,identity_fabrication
